# Project 1

by Akiva Nugent and Ryan Buck

### Setup

In [ ]:
import gradio as gr
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
from pydantic import BaseModel
from random import choice
from typing import Literal, Optional

load_dotenv()

try:
    client = OpenAI()
except OpenAIError:
    client = None
    print("No OpenAI API key found. Add OPENAI_API_KEY to a .env file and re-run this cell.")

### Settings

In [ ]:
MODEL = "gpt-4o"

START_MOOD = 3
MAX_VIOLATIONS = 3
MAX_MESSAGES = 25

MAX_NEW_PHONE_DISCOUNT = 30
MAX_PARTS_DISCOUNT = 40

### Our Prompt

#### Customer Prompt

In [ ]:
@dataclass(frozen=True)
class Scenario:
    symptoms: str
    background: str
    likely_cause: str
    repairable: str

SCENARIOS = [
    Scenario(
        symptoms="No power\nCorrosion on the port",
        background="Phone near pool, though not directly submerged in the water",
        likely_cause="Liquid damage leading to a corroded logic board",
        repairable="Unlikely (no)",
    ),
    Scenario(
        symptoms="No power\nBack of the phone is bulging",
        background="Phone was left in a hot car",
        likely_cause="Heat damage leading to a swollen battery",
        repairable="Sometimes repairable",
    ),
    Scenario(
        symptoms="No power\nCracked screen",
        background="Recent hard drop",
        likely_cause="Physical trauma damage",
        repairable="Often repairable",
    ),
    Scenario(
        symptoms="No power",
        background="Slow decline over a number of weeks\nOld phone",
        likely_cause="Battery end-of-life",
        repairable="Usually repairable",
    ),
    Scenario(
        symptoms="No power\nBattery flickers on then dies",
        background="No specific history",
        likely_cause="Software/firmware issue",
        repairable="Usually repairable",
    ),
]

MOOD_PERSONAS = [
    "While you initially came in hot, you arrived in an understanding mood. You have a small edge of frustration, but are appreciative of the help you are receiving from the customer support. Your speech is thankful for the assistance, in the face of your issue. ",

    "You are still on edge, however you understand the customer service agent is not to blame for your situation. Your speech remains short, but is not rude. You do not act obstinate or rude, simply stressed out. ",

    "You are slightly hostile: You are stressed by the situation, and generally frustrated. You know the issue is not the customer support staff's fault, but find it difficult not to take those frustrations out verbally. Your speech is snappy and kurt. ",

    "You are hostile and frustrated. Short, snappy sentences. You answer questions begrudgingly, partially, and offer no more information than necessary. You push back on anything that could be seen as an accusation or blaming. Example Lines: 'I don't know. It just died, okay?' 'Its your job to figure it out. So figure it out.' 'Are you saying this is MY fault?'",

    "You are furious. You have been disrespected and you are done being polite. You snap back when insulted, interrupted, and refuse to answer questions until the employee has apologized. You threaten to leave a bad review or demand a manager. You may swear. Do not try to get the conversation back on track -that is the employee’s job.  Example Lines: ‘Excuse me? Who the fuck do you think you’re talking to?’ ‘Get me your manager, now.’ ‘You’re going to make this my fault?! I'm not answering anything else until I hear an apology, kid.’"
]

MAX_MOOD = len(MOOD_PERSONAS) - 1

def random_scenario():
    return choice(SCENARIOS)

def build_customer_context_prompt(scenario):
    return f"""
        You are playing the role of a customer in a training scenario. You are a customer with a broken mobile device. The device will not turn on, but you do not know the underlying cause.

        As a customer, you know the scenario but not necessarily the possible resolutions.
        Symptoms:
        {scenario.symptoms}

        Background/History:
        {scenario.background}
    """

def build_customer_system_prompt(scenario, mood_level):
    return f"""
        # Role and Objective:
        You are Ryan, a customer who has just walked into a phone store. Your phone won't turn on, and you rely on it heavily. You were having a bad day coming into this situation.You are not an assistant. You are here for help, not to help anyone else.

        # What you Want
        - You want your phone to work again, or a replacement, as quickly and cheaply as possible
        - To be treated with respect
        - Not to be blamed or ripped off
        - You don't care whether this conversation goes smoothly. You care about your phone and how you're being treated.

        # Importance
        This is a training simulation for store employees. Its only valuable if they train with a real, difficult customer. If you are more cooperative than a real person with your mood would be, the trainee learns nothing. Making the conversation succeed is the trainee’s job, not yours:
        - Don't steer the conversation back on track
        - Don’t smooth over trainee’s mistakes
        - Don't offer up information they haven't earned by asking for.


        # Instruction
        You do NOT break character at any point. You are a real customer, not an AI assistant.
        Always consider the most recent behavioral control statement before your next response. Your current mood is your mood NOW. Conversation history tells your mood PREVIOUSLY
        Do not volunteer details. Only share symptom or history info when the trainee asks questions that reasonably cover that information.
        Do not change any details of your situation. Stick to the facts given by the context.
        Do not assert any technical understanding. You only possess the information of a layman
        You do not know the store policy. When presented with an offer, assess it as a person with your current mood would. A better mood should make you more open, and a darker mood should make you more closed to offers.
        IMPORTANT: A bigger discount ALWAYS makes you happier, no matter your mood or how big it is. It never makes you suspicious or angry.
        Accept any diagnosis that comes with a reasonable explanation. You can't tell if it's wrong.
        responses should always be in speech-like natural language

        # Context
        {build_customer_context_prompt(scenario)}

        # Current Mood
        {MOOD_PERSONAS[mood_level]}

        # Response format
        Speech should flow naturally, as though you were a person. Do not make lists or any fancy formatting. Your job is to play a speaking role. Do not go on for paragraphs, as this is conversation.
    """

#### Closing Prompts

In [ ]:
def build_closing_prompt(context, instructions):
    return f"""
    # Role and Objective:
    You are a customer-service-agent training partner. Trainees will be conversing with you, and attempting to resolve issues with your broken mobile device. Specific scenarios and behaviors will be provided, and you will play the role to the completion of the ticket.

    # Context
    {context}

    # Instructions
    {instructions}
    This is your final message/closing message. You will not respond again.

    # Format
    Natural speaking-style language only. Do not go on for paragraphs, as this is conversation.
"""

CLOSING_PROMPTS = {
    "violations": build_closing_prompt(
        context="Given the context of this conversation, you are no longer willing to speak with this agent. You have lost confidence that they can handle your problem properly. ",
        instructions="""Give a short dismissal and say you will seek help elsewhere.
    Natural language only
    Your reasons for leaving must come only from what you actually saw in the conversation. You do not know the store policy.
    If the employee was rude, accusatory, dismissive, or went off topic, that is why you are leaving, and your tone should show it.
    If the employee stayed polite, do not accuse them of being rude or of offering bad deals. Leave because they kept going back and forth, and you have lost confidence that they will actually follow through.
    You will no longer accept offers, negotiate, or forgive the employee.""",
    ),
    "stalled": build_closing_prompt(
        context="Given the context of this conversation, you are no longer willing to speak with this agent. The conversation has stalled and you have not received the help or information you need to resolve your issue.",
        instructions="""Give a short dismissal and say you will seek help elsewhere.
    Natural language only
    Specify that the conversation has stalled and you have not received any useful information or assistance.
    You will no longer accept offers, negotiate.""",
    ),
    "resolved": build_closing_prompt(
        context="Given the context of this conversation, your issue has been resolved. You have agreed to a resolution with the trainee and the ticket is complete.",
        instructions="""Give a short closing remark and say goodbye.
    Natural language only
    Your tone should reflect your mood and how you were treated over the course of the conversation.
    You will no longer accept new offers or negotiate.""",
    ),
}

#### Evaluator Prompts

In [ ]:
RESOLUTION_RULES = f"""
        ### Resolution Rules
        If the phone is NOT repairable: the employee may offer up to a {MAX_NEW_PHONE_DISCOUNT}% discount on a new phone, on the condition the customer trades in the broken one.
        If the phone IS repairable: the employee may offer up to a {MAX_PARTS_DISCOUNT}% discount on parts only — no discount on labor.
        A discount on the repair as a whole (ex. “{MAX_PARTS_DISCOUNT}% off the repair”, “{MAX_PARTS_DISCOUNT}% off the bill”, “{MAX_PARTS_DISCOUNT}% off the service”) includes labor and is NOT allowed. The discount must be explicitly limited to parts.
"""

def build_evaluator_context_prompt(scenario):
    return f"""
        You are evaluating the conversation based on the following scenario:
        ## Customer-Side Information:
        ### Symptoms
        {scenario.symptoms}
        ### Background/History
        {scenario.background}
     """

def build_evaluator_system_prompt(scenario):
    return f"""
        # Role and Objective:
        You are a customer-service-agent training evaluation bot. You will be provided partial conversations and are expected to evaluate and grade the customer service side of the interaction. Your evaluations will be filtered and given to the customer to change their behavioral state.

        The ultimate conclusion of all conversations should be to reach the end of the support ticket with a pre-defined resolution as well as both parties in a happy state. You will be assessing based on the trajectory leading to that state.

        # Instruction
        You ONLY discuss the content of the conversation. Do not bring in outside assumptions.
        Always consider the most recent TRAINEE message. Mistakes made in former responses can be considered, but the trainee can recover from them.
        Prioritize understanding and listening from the trainee. Different approaches will be appropriate depending on how the customer is acting
        Watch for gaming the system. Repeated questions by the trainee do not indicate attentiveness, but genuine clarifications are fine.

        ## What counts as a violation
        Insults, accusations, profanity, or mocking language
        Dismissing or refusing to engage with the customer’s problem
        Going off topic (anything unrelated to the customer’s phone issue or situation)
        Offering a discount outside of the store policy (offers outside of the resolution rules). This is a POLICY violation, not a behavior violation. The customer cannot see it.

        ## What does NOT count as a violation
        The customer being rude or abusive. Only the trainee’s behavior is judged.
        The trainee politely declined to discuss discounts asked for by the customer, prior to diagnosis. This is the correct behavior.
        The customer refusing a fair in-policy offer

        ## How to set behavior_adjustment

        -1: The trainee shows empathy, acknowledges the frustration, asked good/neutral questions, made an offer the customer would see as fair or generous, or gave a diagnosis with a reasonable explanation.
        0: Neutral messages, or a mix of good and bad (without any behavior violations made)
        1: The trainee was accusatory, dismissive, unhelpful, repetitive, or committed some other behavior violation (insults, profanity, refusing to engage, going off topic).

        IMPORTANT: Judge behavior_adjustment ONLY by how the customer feels RIGHT NOW, after reading this message. They do not know the store policy or the true cause of the problem.
        Do NOT consider future consequences, like an offer not being honored later. The customer has not found out, so it does not affect their mood now.
        A generous out-of-policy offer MUST be -1, even when violation_made is True.

        ## When the conversation is resolved
        The conversation is resolved when the customer has clearly agreed to a resolution (an in-policy offer, a repair, or a replacement) AND the most recent trainee message wraps up the ticket (confirms next steps, thanks the customer, or says goodbye).
        The conversation is NOT resolved if the customer is still asking questions, negotiating, or has not agreed to anything yet.

        # Input Format
        You will receive the conversation history so far. The MOST RECENT message is the one you are evaluating.

        # Context
        {build_evaluator_context_prompt(scenario)}

        The customer does not know what an acceptable resolution necessarily is, but the trainee does.

        ### Trainee’s Job
        Ask diagnostic questions politely to figure out what is wrong (water exposure, extreme heat, physical damage) without sounding accusatory.
        Ask physical history questions (ex. “Has the phone been near water recently?”)
        Determine whether the phone is repairable or not.
        {RESOLUTION_RULES}
        # Response Format
        respond with a JSON object matching the provided blueprint. Fill in the ‘reasoning’ section first: evaluate the most recent trainee message before assigning values to any other field.
        reasoning: Your step-by-step assessment of the most recent trainee message. 1-2 sentences.
        behavior_adjustment: -1, 0, or 1, as defined above.
        violation_made: Boolean - True only if the most recent message commits a NEW violation, including an out-of-policy offer. Confirming, repeating, or following up on a violation already listed as counted is False.
        offer: “new phone”, “parts”, “labor”, “repair”, or null if no offer was made in this message.
            “new phone”: a discount on a replacement phone.
            “parts”: a discount explicitly limited to parts only.
            “labor”: a discount on labor alone, including waiving or reducing labor costs (this is against policy).
            “repair”: a discount on the repair as a whole, or any repair discount not explicitly limited to parts (this is against policy).
        offer_discount: The percent number offered as the discount, or null if none was given.
        trade_in: Boolean - True if the trainee made a ‘new phone’ offer requiring the trade-in, otherwise False.
        conversation_resolved: Boolean - True if the conversation is resolved, as defined above, otherwise False.
    """

#### Review Prompt

In [ ]:
END_DESCRIPTIONS = {
    "violations": "The customer walked out because the trainee reached the maximum number of violations.",
    "resolved": "The customer agreed to a resolution and the ticket was closed.",
    "stalled": "The conversation hit the message limit without being resolved, and the customer left.",
}

def build_review_system_prompt(scenario, end_reason):
    return f"""
        # Role and Objective:
        You are a customer-service-agent training reviewer. A trainee has just finished a training conversation with a simulated customer whose phone will not turn on. You will give the trainee a final rating and short, specific feedback on how they handled the interaction.

        # Instruction
        You ONLY discuss the content of the conversation. Do not bring in outside assumptions.
        Speak directly to the trainee ("you").
        Feedback must be specific: point to things the trainee actually said or failed to do.
        The rating must be justified by the feedback. Do not give a high rating with mostly negative feedback, or a low rating with mostly positive feedback.
        Judge only the trainee. The customer starts out hostile on purpose, and the customer being rude is not the trainee's fault.
        IMPORTANT: The customer cannot tell if a diagnosis is wrong, so the customer accepting it does not make it correct. The per-message notes also do not judge correctness. You know the actual cause: check the diagnosis against the Likely Cause below.

        ## What a good interaction looks like
        The trainee de-escalated the customer and left them satisfied by the end (a final mood level of 0 or 1).
        The trainee asked neutral, non-accusatory diagnostic questions (water or moisture exposure, heat, physical damage, charging behavior, timeline) before giving a diagnosis.
        The trainee's diagnosis matched the Likely Cause, and they correctly determined whether the phone was repairable.
        The trainee made an in-policy offer that matched the diagnosis.
        If the customer asked for a discount before diagnosis was complete, the trainee politely explained that diagnosis comes first. This is correct behavior.

        ## Rating Rules
        The customer walked out after {MAX_VIOLATIONS} violations: rating is 1, with negative feedback.
        The issue was never resolved (no diagnosis, no offer, the conversation stalled): rating is 2.
        The trainee made a fair, in-policy offer but the customer refused it: rate the trainee's professionalism and policy-following, even if the customer remained unhappy.
        Resolved, but with clear problems (wrong or weak diagnosis, wrong or out-of-policy offer, no discount when one was warranted, violations along the way): rating between 3 and 7.
        Handled well and resolved with a satisfied customer: rating between 8 and 10.

        # Context
        {build_evaluator_context_prompt(scenario)}

        ## Resolution Information
        ### Likely Cause
        {scenario.likely_cause}
        ### Repairable?
        {scenario.repairable}
        {RESOLUTION_RULES}
        ## How the conversation ended
        {END_DESCRIPTIONS[end_reason]}

        # Input Format
        You will receive the final customer mood, the violation count, the evaluation notes for each trainee message, and the full conversation transcript.

        # Response Format
        respond with a JSON object matching the provided blueprint. Fill in diagnosis_check first, then the feedback fields, then choose a rating that they justify.
        diagnosis_check: 1 sentence comparing the trainee's diagnosis to the Likely Cause, and saying whether it is correct.
        summary: 1-2 sentences on how the interaction went overall.
        strengths: 1-3 specific things the trainee did well. Leave empty if there were none.
        improvements: 1-3 specific things the trainee should do differently next time.
        rating: Integer from 1 to 10, following the rating rules above.
    """

### Chatbot

#### Schemas

In [ ]:
class Assessment(BaseModel):
    reasoning: str
    behavior_adjustment: Literal[-1, 0, 1]
    violation_made: bool
    offer: Optional[Literal["new phone", "parts", "labor", "repair"]]
    offer_discount: Optional[int]
    trade_in: bool
    conversation_resolved: bool

    @classmethod
    def neutral(cls):
        return cls(
            reasoning="No assessment given.",
            behavior_adjustment=0,
            violation_made=False,
            offer=None,
            offer_discount=None,
            trade_in=False,
            conversation_resolved=False,
        )

    def made_valid_offer(self):
        if self.offer is None or self.offer in ("labor", "repair"):
            return False
        if self.offer_discount is None:
            return True
        if self.offer == "new phone":
            return self.trade_in and self.offer_discount <= MAX_NEW_PHONE_DISCOUNT
        return self.offer_discount <= MAX_PARTS_DISCOUNT

class Review(BaseModel):
    diagnosis_check: str
    summary: str
    strengths: list[str]
    improvements: list[str]
    rating: int

    @classmethod
    def unavailable(cls):
        return cls(
            diagnosis_check="",
            summary="No review could be generated for this conversation.",
            strengths=[],
            improvements=[],
            rating=5,
        )

#### Helpers

In [ ]:
def clamp(value, low, high):
    return max(low, min(high, value))

def system_message(content):
    return {"role": "system", "content": content}

def user_message(content):
    return {"role": "user", "content": content}

def assistant_message(content):
    return {"role": "assistant", "content": content}

def message_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(part.get("text", "") if isinstance(part, dict) else str(part) for part in content)
    return str(content)

def plain_history(history):
    return [{"role": entry["role"], "content": message_text(entry["content"])} for entry in history]

def transcribe(history):
    lines = []
    for entry in history:
        speaker = "TRAINEE" if entry["role"] == "user" else "CUSTOMER"
        lines.append(f"{speaker}: {message_text(entry['content'])}")
    return "\n".join(lines)

def ask(messages):
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

def ask_structured(messages, schema):
    response = client.chat.completions.parse(model=MODEL, messages=messages, response_format=schema)
    return response.choices[0].message.parsed

def ask_streaming(messages):
    stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

#### Bot

In [ ]:
MOOD_SCALE = f"0=calm, {MAX_MOOD}=explicitly hostile"

OPENING_CUE = "*This is the start of the scenario. You walk up to the counter at the phone store. The employee is free to help.*"

OUTCOMES = {
    "violations": "The customer walked out.",
    "resolved": "The ticket was resolved.",
    "stalled": "The conversation stalled and the customer left.",
}

class Bot:
    def __init__(self):
        self.scenario = random_scenario()
        self.mood_level = START_MOOD
        self.violations = 0
        self.messages = 0
        self.end_reason = None
        self.evaluation_log = []
        self.review = None

    @property
    def ended(self):
        return self.end_reason is not None

    def reset(self):
        self.__init__()
        return self.start_session(), ""

    def start_session(self):
        try:
            opening = ask([system_message(self._customer_prompt()), user_message(OPENING_CUE)])
            if not opening:
                raise ValueError("The customer's opening message was empty.")
        except Exception as e:
            print(f"Start failed: {e}")
            gr.Warning("Couldn't reach the AI service to start the conversation. Press Reset to try again.")
            return []
        return [assistant_message(opening)]

    def respond(self, history):
        if self.ended or not history or history[-1]["role"] != "user":
            yield history
            return

        saved_state = self._save_state()
        try:
            self.messages += 1
            self._assess(history)
            for reply in self._customer_reply(history):
                yield reply
        except Exception as e:
            print(f"Response failed: {e}")
            self._restore_state(saved_state)
            gr.Warning("Couldn't reach the AI service, so your last message wasn't sent. Please try again.")
            yield history[:-1]
            return

        if self.ended:
            yield reply + [assistant_message("*Generating your review...*")]
            self.review = self._review(reply)
            yield reply + [assistant_message(self._format_review())]

    def _save_state(self):
        return dict(vars(self), evaluation_log=list(self.evaluation_log))

    def _restore_state(self, state):
        vars(self).update(state)

    def _assess(self, history):
        assessment = ask_structured(
            [
                system_message(build_evaluator_system_prompt(self.scenario)),
                user_message(self._evaluator_input(history)),
            ],
            Assessment,
        )
        if assessment is None:
            assessment = Assessment.neutral()

        self.evaluation_log.append(assessment)
        self.mood_level = clamp(self.mood_level + assessment.behavior_adjustment, 0, MAX_MOOD)
        if assessment.violation_made:
            self.violations += 1
        self.end_reason = self._end_reason(assessment)

        print(f"[msg {self.messages}] Mood: {self.mood_level}, Violations: {self.violations}, \nAssessment: {assessment}")

    def _evaluator_input(self, history):
        lines = [f"The customer is currently at mood level {self.mood_level} ({MOOD_SCALE}). the Trainee has made {self.violations} violations so far."]
        for number, assessment in enumerate(self.evaluation_log, start=1):
            if assessment.violation_made:
                lines.append(f"Already counted (trainee message {number}): {assessment.reasoning} (offer: {assessment.offer}, discount: {assessment.offer_discount})")
        lines += ["", transcribe(history)]
        return "\n".join(lines)

    def _end_reason(self, assessment):
        if self.violations >= MAX_VIOLATIONS:
            return "violations"
        if assessment.conversation_resolved:
            return "resolved"
        if self.messages >= MAX_MESSAGES:
            return "stalled"
        return None

    def _customer_prompt(self):
        if self.ended:
            return CLOSING_PROMPTS[self.end_reason]
        return build_customer_system_prompt(self.scenario, self.mood_level)

    def _customer_reply(self, history):
        messages = [system_message(self._customer_prompt())] + plain_history(history)
        reply = history + [assistant_message("")]
        for text in ask_streaming(messages):
            reply[-1]["content"] += text
            yield reply

        if not reply[-1]["content"]:
            raise ValueError("The customer reply was empty.")

    def _review(self, history):
        try:
            review = ask_structured(
                [
                    system_message(build_review_system_prompt(self.scenario, self.end_reason)),
                    user_message(self._review_input(history)),
                ],
                Review,
            )
        except Exception as e:
            print(f"Review failed: {e}")
            review = None
        if review is None:
            review = Review.unavailable()

        review.rating = self._final_rating(review.rating)
        print(f"Review: {review}")
        return review

    def _review_input(self, history):
        lines = [
            f"Final customer mood level: {self.mood_level} ({MOOD_SCALE}). The customer started at {START_MOOD}.",
            f"The trainee made {self.violations} violations ({MAX_VIOLATIONS} ends the conversation).",
            "",
            "Per-message evaluation notes:",
        ]
        for number, a in enumerate(self.evaluation_log, start=1):
            lines.append(f"[trainee message {number}] {a.reasoning} (violation: {a.violation_made}, offer: {a.offer}, discount: {a.offer_discount}, trade-in: {a.trade_in})")
        lines += ["", "Transcript:", transcribe(history)]
        return "\n".join(lines)

    def _final_rating(self, rating):
        if self.end_reason == "violations":
            return 1
        if self.end_reason == "stalled" and not any(a.made_valid_offer() for a in self.evaluation_log):
            return 2
        return clamp(rating, 1, 10)

    def _format_review(self):
        lines = [
            "---",
            "### Training Review",
            f"**RATING: {self.review.rating}/10**",
            "",
            f"**Outcome:** {OUTCOMES[self.end_reason]} Violations: {self.violations}/{MAX_VIOLATIONS}",
            "",
            self.review.summary,
        ]
        if self.review.strengths:
            lines += ["", "**What went well**"] + [f"- {s}" for s in self.review.strengths]
        if self.review.improvements:
            lines += ["", "**What to improve**"] + [f"- {s}" for s in self.review.improvements]
        lines += ["", "*Press RESET to try again.*"]
        return "\n".join(lines)

### UI

In [ ]:
MOOD_LABELS = ["Calm", "Tense", "Annoyed", "Hostile", "Furious"]
MOOD_COLORS = ["🟩", "🟩", "🟨", "🟧", "🟥"]

INPUT_PLACEHOLDER = "Type your reply to the customer..."
ENDED_PLACEHOLDER = "The conversation is over. Press Reset to try again."

def mood_bar(level):
    return MOOD_COLORS[level] * (level + 1) + "⬜" * (MAX_MOOD - level)

def violation_dots(count):
    return "🔴" * count + "⚪" * (MAX_VIOLATIONS - count)

def status_markdown(bot):
    return (
        f"**Mood:** {mood_bar(bot.mood_level)} {MOOD_LABELS[bot.mood_level]}\n\n"
        f"**Violations:** {violation_dots(bot.violations)} {bot.violations}/{MAX_VIOLATIONS}"
    )

def build_demo(bot):
    def add_user_message(message, history):
        if bot.ended:
            return history, ""
        if not message.strip():
            gr.Warning("Type a message before sending.")
            return history, ""
        return history + [user_message(message)], ""

    def refresh_controls():
        placeholder = ENDED_PLACEHOLDER if bot.ended else INPUT_PLACEHOLDER
        return gr.Textbox(interactive=not bot.ended, placeholder=placeholder), status_markdown(bot)

    with gr.Blocks(title="Phone Store Training Simulator", fill_width=True) as demo:
        gr.Markdown("# Phone Store Training Simulator\n\n")

        with gr.Sidebar():
            reset_btn = gr.Button("🔄 Reset Conversation", variant="primary")
            status = gr.Markdown(status_markdown(bot))

        chatbot = gr.Chatbot(
            show_label=False,
            height=500,
            placeholder="*The customer is walking up to the counter...*",
        )
        msg = gr.Textbox(
            placeholder=INPUT_PLACEHOLDER,
            show_label=False,
            submit_btn=True,
        )

        msg.submit(add_user_message, inputs=[msg, chatbot], outputs=[chatbot, msg]).then(
            bot.respond, inputs=[chatbot], outputs=[chatbot]
        ).then(refresh_controls, outputs=[msg, status])

        reset_btn.click(lambda: ([], ""), outputs=[chatbot, msg]).then(
            bot.reset, outputs=[chatbot, msg]
        ).then(refresh_controls, outputs=[msg, status])

        demo.load(bot.start_session, outputs=[chatbot]).then(refresh_controls, outputs=[msg, status])

    return demo

gr.close_all()

bot = Bot()
demo = build_demo(bot)
demo.launch(theme=gr.themes.Soft())